# AirGreenOps dashboard (Google Colab)

Airport performance under comparable conditions (RQ2) and emission reduction scenarios without extra delay (RQ3).
The data stays in Databricks: this notebook reads the result tables of notebooks 05 and 06 through the Databricks SQL
warehouse, so the Spark cluster does not need to run. The tabs use the same code as the Databricks dashboard
(`src/thesis_dashboard.py`). The EM06DS prediction tabs score flights with Spark ML models, so they stay in notebook 03.

**Before running:** in Databricks, create an access token (profile icon > Settings > Developer > Access tokens).
In Colab, open Secrets (key icon on the left), add it with the name `DATABRICKS_TOKEN` and allow notebook access.
Never paste the token into a cell: this notebook is public on GitHub.

Then Runtime > Run all. The last cell shows the dashboard and prints a public link (valid while this notebook runs).

In [ ]:
%pip install -q gradio==4.44.1 huggingface_hub==0.25.2 "databricks-sql-connector[pyarrow]==4.6.0"

In [ ]:
# Project code from GitHub: scenario calculator and dashboard tabs (src/)
import os
import subprocess
import sys

REPO_DIR = os.environ.get("AIRGREENOPS_REPO", "/content/green-airport-operations-bigdata")
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/mhalraed/green-airport-operations-bigdata.git", REPO_DIR],
                   check=True)
sys.path.insert(0, f"{REPO_DIR}/src")

In [ ]:
# Connection to Databricks: the data stays in Databricks; each query runs on the SQL warehouse and returns a pandas table.
# The access token is read from Colab Secrets (name DATABRICKS_TOKEN).
import os
from databricks import sql as databricks_sql

try:
    from google.colab import userdata
    TOKEN = userdata.get("DATABRICKS_TOKEN")
except ImportError:  # outside Colab: environment variable
    TOKEN = os.environ["DATABRICKS_TOKEN"]

connection = databricks_sql.connect(server_hostname="dbc-621d6bfc-6584.cloud.databricks.com",
                                    http_path="/sql/1.0/warehouses/e1e7ec55e00b3788",
                                    access_token=TOKEN, catalog="new", schema="airgreenops")


def sql(query):
    with connection.cursor() as cursor:
        cursor.execute(query)
        return cursor.fetchall_arrow().to_pandas()


def read(table):
    return sql(f"SELECT * FROM {table}")


print("Connected to Databricks:", len(sql("SHOW TABLES")), "tables in new.airgreenops")

In [ ]:
# Results of notebooks 05 and 06; stops if the scenario calculator does not reproduce notebook 06
import thesis_dashboard as thesis

THESIS_DATA = thesis.load(read)

In [ ]:
# Dashboard
import gradio as gr

with gr.Blocks(title="AirGreenOps Dashboard") as demo:
    gr.Markdown("# AirGreenOps: airport ground emissions\n"
                "Results for 76 major U.S. airports (BTS on-time data 2023-2025, ICAO engine databank fuel flow).")
    thesis.add_tabs(gr, demo, THESIS_DATA)

demo.launch(share=True, debug=False)